In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import flopy
import shutil
import tempfile
from pathlib import Path 


In [ ]:
MODFLOW_BIN_DIR = "./usgs_code"
# 1. Define the exact path where your Mac downloaded the binaries
MF6_EXE = os.path.join(MODFLOW_BIN_DIR, "mf6")
MP7_EXE = os.path.join(MODFLOW_BIN_DIR, "mp7")

NameError: name 'os' is not defined

In [ ]:
# Run only if the mf6, mp7, and mt3dusgs binaries aren't in the folder "usgs_code"

# # Get MODFLOW 6
# flopy.utils.get_modflow(subset="mf6", bindir=MODFLOW_BIN_DIR)

# # Get MODPATH 7 
# flopy.utils.get_modflow(subset="mp7", bindir=MODFLOW_BIN_DIR)

# # Get MT3D-USGS
# flopy.utils.get_modflow(subset="mt3dusgs", bindir=MODFLOW_BIN_DIR)

In [ ]:
# Set up a temporary directory
temp_dir = Path(tempfile.mkdtemp())
sim_name = "test_mf6"

try:
    # 1. Initialize Simulation
    sim = flopy.mf6.MFSimulation(
        sim_name=sim_name, 
        version="mf6", 
        exe_name=f"{MODFLOW_BIN_DIR}/mf6", 
        sim_ws=str(temp_dir)
    )

    # 2. Add Timing & Solver
    tdis = flopy.mf6.ModflowTdis(sim, time_units="DAYS", nper=1, perioddata=[(1.0, 1, 1.0)])
    ims = flopy.mf6.ModflowIms(sim, complexity="SIMPLE")

    # 3. Add Groundwater Flow (GWF) Model
    gwf = flopy.mf6.ModflowGwf(sim, modelname=sim_name, save_flows=True)
    dis = flopy.mf6.ModflowGwfdis(gwf, nlay=1, nrow=3, ncol=3, delr=10.0, delc=10.0, top=10.0, botm=0.0)
    npf = flopy.mf6.ModflowGwfnpf(gwf, k=1.0)
    ic = flopy.mf6.ModflowGwfic(gwf, strt=10.0)
    chd = flopy.mf6.ModflowGwfchd(gwf, stress_period_data=[((0, 0, 0), 10.0), ((0, 2, 2), 5.0)])

    # 4. Write inputs and run simulation
    sim.write_simulation()
    success, buff = sim.run_simulation(silent=True)

    if success:
        print("MODFLOW 6 is installed and ran successfully with FloPy!")
    else:
        print("Model failed to run. Output log:")
        print("\n".join(buff))

finally:
    # Clean up files
    shutil.rmtree(temp_dir)